# M02 — 狀態與 Reducer

本 notebook 對應同資料夾的 `README.md`，逐格執行即可。

主軸：LangGraph 更新 state 的預設行為是「覆蓋」。我們先示範它何時會出錯，
再用 reducer（`operator.add` / `add_messages`）讓「累積」這個特殊情況消失。
全程沿用 M01 學過的 `StateGraph` / `add_node` / `add_edge`，只多了一個型別標註。

## 1. 環境準備

載入共用 helper。本模組多數範例是「純圖」邏輯、不一定需要 LLM，
但最後會用 `MessagesState` 跑一個真的會呼叫模型的小聊天圖，所以一併備好 model。

In [ ]:
# Load shared helpers so every notebook stays provider-agnostic.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1] / "_shared"))
from course_utils import get_model, load_env

load_env()
model = get_model()

## 2. 先看「覆蓋」問題

我們定義一個 state，裡面有個 `logs: list`，**沒有掛任何 reducer**。
兩個 node 各自往 logs 記一筆。直覺以為最後有兩筆，實際上只會剩最後一筆——
因為第二個 node 回傳的 list 會「整個覆蓋」第一個 node 寫的。

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class OverwriteState(TypedDict):
    logs: list  # No reducer -> default behaviour is "last write wins" (overwrite).


def node_a(state: OverwriteState) -> dict:
    # Each node returns ONLY the part it wants to write.
    return {"logs": ["A wrote a log"]}


def node_b(state: OverwriteState) -> dict:
    return {"logs": ["B wrote a log"]}


builder = StateGraph(OverwriteState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)
overwrite_graph = builder.compile()

result = overwrite_graph.invoke({"logs": []})
print(result)
# Expected output: {'logs': ['B wrote a log']}
# Note: "A wrote a log" is GONE. node_b's dict overwrote node_a's list entirely.

## 3. 加上 reducer：改成累加

解法不是「在每個 node 手動 `state["logs"] + [...]`」（那是到處貼樣板）。
而是改變這個鍵的**合併規則**：把型別標註成 `Annotated[list, add]`。
reducer `operator.add` 對 list 就是串接，所以兩個 node 的結果會累積起來。

注意：node 的程式碼**一字未改**，只動了 state 的型別標註。

In [ ]:
from typing import Annotated
from operator import add


class AccumulateState(TypedDict):
    logs: Annotated[list, add]  # add reducer -> new value is appended, not overwritten.


# Same two nodes as before — they still only return "the part I add".
def acc_node_a(state: AccumulateState) -> dict:
    return {"logs": ["A wrote a log"]}


def acc_node_b(state: AccumulateState) -> dict:
    return {"logs": ["B wrote a log"]}


builder = StateGraph(AccumulateState)
builder.add_node("node_a", acc_node_a)
builder.add_node("node_b", acc_node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)
accumulate_graph = builder.compile()

result = accumulate_graph.invoke({"logs": []})
print(result)
# Expected output: {'logs': ['A wrote a log', 'B wrote a log']}
# Both logs survive — the reducer did the appending, the nodes did not.

## 4. 多鍵、不同 reducer、部分更新

一個 state 裡不同的鍵可以有不同合併規則。而且 node 只要回傳「它想改的鍵」，
沒回傳的鍵會自動保持原值（partial update）。

下面 `counter` 用 add 累加（數字相加）、`logs` 用 add 串接、`status` 預設覆蓋。

In [ ]:
class MultiKeyState(TypedDict):
    counter: Annotated[int, add]  # add on ints means SUM.
    logs: Annotated[list, add]    # add on lists means concatenation.
    status: str                   # No reducer -> overwrite (latest wins).


def stepper(state: MultiKeyState) -> dict:
    # Returns a partial update: bumps counter, appends a log, sets status.
    return {"counter": 1, "logs": ["step done"], "status": "running"}


def finalizer(state: MultiKeyState) -> dict:
    # Only touches counter, logs, status — same keys, different intent.
    return {"counter": 1, "logs": ["finalized"], "status": "done"}


builder = StateGraph(MultiKeyState)
builder.add_node("stepper", stepper)
builder.add_node("finalizer", finalizer)
builder.add_edge(START, "stepper")
builder.add_edge("stepper", "finalizer")
builder.add_edge("finalizer", END)
multi_graph = builder.compile()

result = multi_graph.invoke({"counter": 0, "logs": [], "status": "init"})
print(result)
# Expected output:
# {'counter': 2, 'logs': ['step done', 'finalized'], 'status': 'done'}
# counter: 0 + 1 + 1 = 2 (summed). logs concatenated. status overwritten to last value.

## 🧪 練習 1

在 `MultiKeyState` 再加一個欄位 `errors: Annotated[list, add]`，
讓兩個 node 各自往 `errors` 記一筆（例如 `{"errors": ["stepper warn"]}`）。
預期執行後 `errors` 會是兩筆而非一筆。

再試一個對照：把 `errors` 改成沒有 reducer 的純 `list`，重跑一次，
觀察是不是又退回「只剩最後一筆」的覆蓋行為。這就是 reducer 的全部魔法。

## 5. 對話訊息：`add_messages` 與 `MessagesState`

聊天場景幾乎都有一個 `messages` 欄位要累積。用 `operator.add` 雖能串接，
但 `add_messages` 更聰明：會把 dict 形式的訊息自動轉成 Message 物件，
並依訊息 id 去重 / 更新（同 id 是「取代」而非「重複加一筆」）。

先看它怎麼自己掛在 state 上（這正是內建 `MessagesState` 的內容）。

In [ ]:
from langgraph.graph.message import add_messages


class ManualChatState(TypedDict):
    messages: Annotated[list, add_messages]  # This single line IS MessagesState.


def echo_node(state: ManualChatState) -> dict:
    # Return ONLY the new message; add_messages appends it to history.
    last_text = state["messages"][-1].content
    from langchain.messages import AIMessage
    return {"messages": [AIMessage(content=f"你剛剛說：{last_text}")]}


builder = StateGraph(ManualChatState)
builder.add_node("echo", echo_node)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)
echo_graph = builder.compile()

# Input as a dict message — add_messages will coerce it into a HumanMessage.
result = echo_graph.invoke({"messages": [{"role": "user", "content": "你好"}]})
for m in result["messages"]:
    print(type(m).__name__, "->", m.content)
# Expected output (2 messages, accumulated):
# HumanMessage -> 你好
# AIMessage -> 你剛剛說：你好

## 6. 用內建 `MessagesState` 接真模型

`MessagesState` 已經內含 `messages: Annotated[list, add_messages]`，
所以聊天圖不必自己宣告。這裡讓 node 真的呼叫 LLM，回傳的 `AIMessage`
由 `add_messages` 自動累積進歷史。

要加自己的欄位時，繼承 `MessagesState` 再補即可。

In [ ]:
from langgraph.graph import MessagesState


def chat_node(state: MessagesState) -> dict:
    # Feed the whole running history to the model; append its reply.
    ai = model.invoke(state["messages"])
    return {"messages": [ai]}


builder = StateGraph(MessagesState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)
chat_graph = builder.compile()

result = chat_graph.invoke(
    {"messages": [{"role": "user", "content": "用一句話介紹 LangGraph"}]}
)
# History now has the user turn + the model's AIMessage, accumulated by add_messages.
print("訊息數量:", len(result["messages"]))   # Expected: 2
print("最後一則:", result["messages"][-1].content)

## 🧪 練習 2

1. 繼承 `MessagesState` 做一個 `ChatWithCounterState`，多加一個
   `turns: Annotated[int, add]` 欄位，讓 `chat_node` 每次回傳時順手 `+1`。
   跑兩次 invoke（把上一輪的 `messages` 回填當輸入），觀察 `turns` 累加、
   `messages` 也持續變長。
2. 進階：手動建立兩則「相同 id」的 `AIMessage` 餵進 `add_messages`，
   確認最終只剩一則（被後者取代），體會它「依 id 去重」和 `operator.add`
   的差別。

## 7. 小結 & 下一步

這一模組只有一個核心動作：**在 state 的鍵上做型別標註 `Annotated[type, reducer]`，
讓「累積」這個特殊情況消失**——node 從此只回傳「新增的部分」，合併交給 reducer。

你練過了：
- 預設「覆蓋」行為，以及它何時是 bug。
- `operator.add`：list 串接、int 求和。
- 多鍵混搭不同 reducer + 部分更新。
- `add_messages` / `MessagesState`：自動累積對話、轉型、依 id 去重。

下一站 **M03 — 條件路由與 Command**：目前圖都是一條直線。M03 讓圖會**分支、會迴圈**，
並用 `Command` 同時「更新 state + 決定下一步去哪」。屆時 Agent 在迴圈裡一輪一輪累積
對話，靠的正是這裡的 `add_messages`。